Les algorithmes du Machine Learning  et le Fight des IA


Phase 0: La mise en route :

Phase A : Prédire les prix immobiliers (régression)

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    r2_score, mean_absolute_error, root_mean_squared_error,
    f1_score
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Les Prix de l'immobilier (en régression)

def charger_immobilier():
    data = fetch_california_housing()
    X, y = data.data, data.target
    print(f"Shape : {X.shape} | Variables : {data.feature_names}")
    print("Cible : prix médian en centaines de milliers de $")
    return X, y

def evaluer_regression(nom, modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{nom:<20} R2={r2:.2f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}

# execution du codes
X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

#rappelle le random state = 42 est une valeur par "defaut", cela permet par exemple à un groupe en fixant 42 de débuter toujours avec un split identique 
# Nous pourrions tous débuté à 26 

evaluer_regression("La régression linéaire",  LinearRegression(),                         X_train_s, X_test_s, y_train, y_test)
evaluer_regression("Le random forest",      RandomForestRegressor(n_estimators=200, random_state=42), X_train_s, X_test_s, y_train, y_test)

# ── Checkpoints qualité Phase A ───────────────────────────────────────────────

# --- Cas limite : 100 lignes ---
X_train_100, X_test_100, y_train_100, y_test_100 = train_test_split(
    X[:100], y[:100], test_size=0.2, random_state=42
)
scaler_100 = StandardScaler().fit(X_train_100)
evaluer_regression("LinReg 100 lignes",
                   LinearRegression(),
                   scaler_100.transform(X_train_100),
                   scaler_100.transform(X_test_100),
                   y_train_100, y_test_100)

# le R2 s'effondre completement, on tombe a des valeurs vraiment mauvaises
# voire négatives parfois. c'est logique en fait : le modele a vu 80 exemples
# pour apprendre les prix de TOUTE la californie, c'est beaucoup trop peu.
# il n'a pas eu assez de diversité dans les données pour capturer les patterns
# (quartiers riches, zones rurales, bord de mer etc...)
# => un modele a besoin de données pour generaiser, 80 lignes c'est rien


# --- Cas adversarial : quartier fictif hors plage ---
# ordre des variables : MedInc, HouseAge, AveRooms, AveBedrms,
#                       Population, AveOccup, Latitude, Longitude
quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 36.5, -119.0]])
quartier_scaled = scaler.transform(quartier_fictif)

# on réutilise le modele déja entrainé sur le dataset complet
pred_lr = LinearRegression().fit(
    scaler.transform(X_train), y_train).predict(quartier_scaled)[0]
print(f"Prédiction LR  : {pred_lr:.2f} (x100k$)")

# la valeur sortie est completement absurde, le modele s'en fout que
# le revenu median soit a 0 ou que la population soit de 9000 personnes,
# il extrapole juste dans le vide sans aucun garde fou.
# en prod ca serait un vrai probleme : on pourrait se retrouver a afficher
# un prix negatif ou un prix de 50 millions a un utilisateur
# ce qu'on devrait faire :
#   - verifier que les valeurs d'entrée sont dans une plage "normale" (celle du train)
#   - rejeter la requete si c'est hors plage et renvoyer une erreur clair
#   - jamais faire confiance a ce que l'utilisateur envoie sans validation

Shape : (20640, 8) | Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible : prix médian en centaines de milliers de $
La régression linéaire R2=0.58  MAE=0.53  RMSE=0.75
Le random forest     R2=0.81  MAE=0.33  RMSE=0.50
LinReg 100 lignes    R2=0.71  MAE=0.38  RMSE=0.49
Prédiction LR  : -0.40 (x100k$)


Phase B: : Segmenter les clients d'AirBnB (non supervisé)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import zipfile

def charger_airbnb(chemin):
    df = pd.read_csv(chemin, encoding='latin-1', low_memory=False,
                     compression='zip' if chemin.endswith('.zip') else None)
    print(f"Brut : {df.shape[0]} lignes")
    df = df[colonnes_features].copy()
    df["price"] = df["price"].replace(r'[\$,]', '', regex=True).astype(float)
    df = df.dropna()
    df = df[df["price"] > 0]
    print(f"Après nettoyage : {df.shape[0]} lignes")
    return df

def choisir_k(X_scaled, k_range=range(2, 9)):
    print(f"\n{'k':>3}  {'inertie':>10}  {'silhouette':>10}")
    print("-" * 28)
    meilleur_k = 2
    meilleure_sil = -1
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_)
        print(f"{k:>3}  {km.inertia_:>10.1f}  {sil:>10.3f}")
        if sil > meilleure_sil:
            meilleure_sil = sil
            meilleur_k = k
    print(f"\n→ k optimal retenu : {meilleur_k} (silhouette={meilleure_sil:.3f})")
    return meilleur_k

Brut : 279712 lignes
Après nettoyage + échantillon : 10000 lignes

  k     inertie  silhouette
----------------------------
  2     32969.1       0.451
  3     27644.1       0.472
  4     22578.6       0.482
  5     18511.9       0.429
  6     16557.6       0.343
  7     14295.4       0.386
  8     12476.0       0.347


In [25]:
CHEMIN_ZIP = "Listings.zip"

colonnes_features = ["price", "minimum_nights", "accommodates",
                     "review_scores_rating", "bedrooms"]

# === CAS NORMAL ===
print("=== CAS NORMAL ===\n")
df_airbnb = charger_airbnb(CHEMIN_ZIP)

# échantillon de 20 000 lignes (279k = trop lent sinon)
df_sample = df_airbnb.sample(n=10_000, random_state=42)

scaler_ab = StandardScaler()
X_scaled  = scaler_ab.fit_transform(df_sample[colonnes_features])

k_optimal = choisir_k(X_scaled)

km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_sample = df_sample.copy()
df_sample["segment"] = km_final.fit_predict(X_scaled)

print(f"\n=== DESCRIPTION DES {k_optimal} SEGMENTS ===\n")
desc = df_sample.groupby("segment")[colonnes_features].mean().round(1)
print(desc)

print("\n--- Interprétation ---")
for seg in range(k_optimal):
    prix  = desc.loc[seg, "price"]
    nuits = desc.loc[seg, "minimum_nights"]
    cap   = desc.loc[seg, "accommodates"]
    note  = desc.loc[seg, "review_scores_rating"]
    if prix > desc["price"].median() * 1.8:
        label = "Premium (prix élevé)"
    elif nuits > 20:
        label = "Longue durée (séjours longs)"
    elif cap >= 4:
        label = "Familial / groupe"
    elif note >= 95:
        label = "Très bien noté / qualité"
    else:
        label = "Budget / standard"
    print(f"  Segment {seg} → {label}  "
          f"(prix={prix:.0f}€, min_nuits={nuits:.0f}, "
          f"capacité={cap:.0f}, note={note:.0f})")

# === CAS LIMITE : sans standardiser ===
print("\n=== CAS LIMITE (sans standardisation) ===\n")
X_brut = df_sample[colonnes_features].values
km_brut = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
labels_brut = km_brut.fit_predict(X_brut)
sil_brut   = silhouette_score(X_brut, labels_brut)
sil_scaled = silhouette_score(X_scaled, km_final.labels_)
print(f"Silhouette SANS standardisation : {sil_brut:.3f}")
print(f"Silhouette AVEC standardisation : {sil_scaled:.3f}")
print("→ 'price' (0–1000) écrase 'bedrooms' (1–5) sans standardisation.")
print("  Les segments n'ont plus de sens métier.")

# === CAS ADVERSARIAL : annonce à 100 000€/nuit ===
print("\n=== CAS ADVERSARIAL (outlier prix=100 000) ===\n")
df_poison = df_sample[colonnes_features].copy()
outlier = pd.DataFrame([[100_000, 1, 2, 100, 1]], columns=colonnes_features)
df_poison = pd.concat([df_poison, outlier], ignore_index=True)

scaler_p = StandardScaler()
X_poison = scaler_p.fit_transform(df_poison)
km_poison = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_poison.fit(X_poison)

print("Centre 'price' AVANT outlier par segment :")
for i, c in enumerate(km_final.cluster_centers_):
    print(f"  Segment {i} : price_scaled={c[0]:.3f}")
print("\nCentre 'price' APRÈS outlier par segment :")
for i, c in enumerate(km_poison.cluster_centers_):
    print(f"  Segment {i} : price_scaled={c[0]:.3f}")
print("\n→ L'outlier déforme les centres. Nettoyage J2 obligatoire avant tout KMeans.")

=== CAS NORMAL ===

Brut : 279712 lignes
Après nettoyage + échantillon : 10000 lignes


KeyError: "['bedrooms'] not in index"